[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdsanch1/simrc/blob/master/01.%20Parte%201/04.%20Clase%204/04Class%20NB.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jdsanch1/SimRC/master?labpath=01.%20Parte%201%2F04.%20Clase%204%2F04Class%20NB.ipynb)

In [ ]:
import importlib, subprocess, sys

_required = ["yfinance", "pandas", "numpy", "matplotlib", "seaborn", "scipy", "sklearn", "statsmodels", "cvxpy"]
_missing  = [pkg for pkg in _required if importlib.util.find_spec(pkg) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _missing)

# Clase 4: Ratio de Sharpe y portafolio óptimo

**[Juan Diego Sánchez Torres](https://www.researchgate.net/profile/Juan_Diego_Sanchez_Torres)**  
*Profesor*, [MAF ITESO](http://maf.iteso.mx/web/general/detalle?group_id=5858156)  
dsanchez@iteso.mx

## Objetivos de aprendizaje

- Calcular el **ratio de Sharpe** y encontrar el portafolio tangente.
- Trazar la **frontera eficiente** de Markowitz mediante QP paramétrico.
- Formular y resolver problemas de optimización con **CVXPY** (enfoque DCP).
- Aplicar la transformación de Cornuejols & Tütüncü para maximizar Sharpe.
- Agrupar activos con **K-Means** y **clustering jerárquico**.

---

## Introducción teórica

### El problema de selección de portafolios

Dado un conjunto de $n$ activos con vector de rendimientos esperados $\boldsymbol{\mu}$ y matriz de covarianza $\boldsymbol{\Sigma}$, el inversionista busca asignar pesos $\mathbf{w} = (w_1, \ldots, w_n)^\top$ que **maximicen el rendimiento ajustado por riesgo**.

El rendimiento y la varianza del portafolio son:

$$
\mu_p = \mathbf{w}^\top \boldsymbol{\mu}, \qquad \sigma_p^2 = \mathbf{w}^\top \boldsymbol{\Sigma} \, \mathbf{w}
$$

### Ratio de Sharpe

El **ratio de Sharpe** (Sharpe, 1966) mide el rendimiento en exceso por unidad de riesgo:

$$
S = \frac{\mu_p - r_f}{\sigma_p}
$$

donde $r_f$ es la tasa libre de riesgo. El **portafolio óptimo** es el que maximiza $S$.

### Frontera eficiente

La **frontera eficiente** (Markowitz, 1952) es el conjunto de portafolios que ofrecen el **máximo rendimiento para cada nivel de riesgo**. Se obtiene resolviendo, para cada rendimiento objetivo $\mu^*$:

$$
\min_{\mathbf{w}} \quad \mathbf{w}^\top \boldsymbol{\Sigma} \, \mathbf{w} \qquad \text{s.a.} \quad \boldsymbol{\mu}^\top \mathbf{w} = \mu^*, \quad \sum_i w_i = 1, \quad w_i \geq 0
$$

Este es un **problema de programación cuadrática** (QP) que resolvemos con CVXPY bajo el enfoque de **Programación Convexa Disciplinada** (DCP) (Boyd & Vandenberghe, 2004).

## 1. Ejemplo introductorio: portafolio de dos activos

Comenzamos con un ejemplo sencillo para ilustrar cómo los pesos afectan el rendimiento y riesgo del portafolio.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import cvxpy as cp
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.cluster import KMeans
import scipy.cluster.hierarchy as hac

sns.set_theme(style="darkgrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 100, "figure.figsize": (12, 5)})
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 8)

In [ ]:
# Rendimientos hipotéticos de dos acciones (5 períodos)
rendimientos = pd.DataFrame({
    "Acción A": [0.10, 0.24, 0.05, -0.02, 0.20],
    "Acción B": [0.15, -0.05, 0.10, 0.08, -0.05]
})

# Pesos iguales
w = np.array([0.5, 0.5])
rend_port = rendimientos.values @ w

total = rendimientos.copy()
total["Portafolio"] = rend_port

print("Desviación estándar:")
print(total.std())
print(f"\nCorrelación entre A y B: {rendimientos.corr().iloc[0,1]:.3f}")

In [ ]:
total.plot(kind="bar", figsize=(10, 5))
plt.title("Rendimientos por período: acciones individuales vs. portafolio")
plt.ylabel("Rendimiento")
plt.xlabel("Período")
plt.tight_layout()

---
## 2. Descarga de datos reales

Descargamos precios de cierre ajustados de 4 acciones usando `yfinance`.

In [ ]:
def get_historical_closes(tickers, start_date, end_date):
    """Descarga precios de cierre ajustados usando yfinance."""
    data = yf.download(tickers, start=start_date, end=end_date,
                       auto_adjust=True, progress=False)
    if isinstance(data.columns, pd.MultiIndex):
        closes = data["Close"]
    else:
        closes = data[["Close"]]
        closes.columns = [tickers] if isinstance(tickers, str) else list(tickers)
    return closes.dropna()

tickers = ["AA", "AAPL", "MSFT", "KO"]
closes = get_historical_closes(tickers, "2025-01-01", "2025-03-27")

fig, ax = plt.subplots()
(closes / closes.iloc[0] * 100).plot(ax=ax)
ax.set_title("Precios normalizados (base 100)")
ax.set_ylabel("Índice")
plt.tight_layout()

---
## 3. Rendimientos, varianza y ratio de Sharpe

### Rendimientos logarítmicos diarios

In [ ]:
daily_returns = np.log(closes / closes.shift(1)).dropna()
daily_returns.describe()

### Correlación entre activos

In [ ]:
corr = daily_returns.corr()
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm",
            vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title("Correlación de rendimientos diarios")
plt.tight_layout()

### Nota teórica: varianza del portafolio

La varianza de un portafolio con pesos $\mathbf{w}$ es:

$$
\sigma_p^2 = \mathbf{w}^\top \boldsymbol{\Sigma} \, \mathbf{w} = \sum_{i=1}^{n} \sum_{j=1}^{n} w_i \, w_j \, \sigma_{ij}
$$

Para pesos iguales ($w_i = 1/n$), esta se simplifica al promedio de todas las covarianzas.

In [ ]:
def calc_portfolio_var(returns, weights=None):
    """Varianza del portafolio."""
    n = returns.columns.size
    if weights is None:
        weights = np.ones(n) / n
    sigma = np.cov(returns.T, ddof=0)
    return (weights @ sigma @ weights)

def sharpe_ratio(returns, weights=None, risk_free_rate=0.015):
    """Ratio de Sharpe del portafolio."""
    n = returns.columns.size
    if weights is None:
        weights = np.ones(n) / n
    var = calc_portfolio_var(returns, weights)
    port_return = returns.mean().values @ weights
    return (port_return - risk_free_rate) / np.sqrt(var)

# Rendimientos anualizados (aproximación)
annual_returns = daily_returns * 252

print(f"Varianza del portafolio (pesos iguales): {calc_portfolio_var(annual_returns):.6f}")
print(f"Ratio de Sharpe (pesos iguales):         {sharpe_ratio(annual_returns, risk_free_rate=0.0003):.4f}")

---
## 4. Optimización con CVXPY (Programación Convexa Disciplinada)

### 4.1 Reglas DCP

**CVXPY** verifica automáticamente la convexidad del problema mediante las reglas DCP (Grant & Boyd, 2008):

1. La función objetivo debe ser **convexa** (al minimizar) o **cóncava** (al maximizar).
2. Las restricciones de igualdad deben ser **afines**.
3. Las restricciones de desigualdad: `convexa ≤ cóncava`.

Expresiones DCP para portafolios:
- `cp.quad_form(w, Σ)` → **convexa** (Σ es PSD)
- `μ @ w` → **afín**
- `cp.sum(w) == 1` → **afín**
- `w >= 0` → **afín**

> Referencia: Boyd, S. & Vandenberghe, L. (2004). *Convex Optimization*. Cambridge University Press.

### 4.2 Maximización del ratio de Sharpe

El ratio de Sharpe no es directamente convexo. Usamos la **transformación de Cornuejols & Tütüncü**: definimos $\mathbf{y} = \mathbf{w}/\kappa$ y resolvemos:

$$
\min_{\mathbf{y}, \kappa} \quad \mathbf{y}^\top \boldsymbol{\Sigma} \, \mathbf{y} \qquad \text{s.a.} \quad (\boldsymbol{\mu} - r_f)^\top \mathbf{y} = 1, \quad \sum y_i = \kappa, \quad \mathbf{y} \geq 0
$$

Luego $\mathbf{w}^* = \mathbf{y}^* / \kappa^*$ recupera los pesos óptimos.

In [ ]:
def optimize_portfolio(returns, risk_free_rate):
    """Maximizar Sharpe via transformación DCP."""
    n = returns.columns.size
    mu = returns.mean().values
    Sigma = np.cov(returns.T, ddof=0)
    excess = mu - risk_free_rate

    y = cp.Variable(n)
    kappa = cp.Variable(nonneg=True)
    objective = cp.Minimize(cp.quad_form(y, Sigma))
    constraints = [excess @ y == 1, cp.sum(y) == kappa, y >= 0]
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.ECOS)

    final_w = y.value / kappa.value
    final_sharpe = sharpe_ratio(returns, final_w, risk_free_rate)
    return (final_w, final_sharpe)

w_opt, sharpe_opt = optimize_portfolio(annual_returns, 0.0003)
print(f"Ratio de Sharpe óptimo: {sharpe_opt:.4f}\n")
print("Pesos óptimos:")
for ticker, w in zip(tickers, w_opt):
    print(f"  {ticker}: {w:.4f}")

### 4.3 Frontera eficiente

Para trazar la frontera, resolvemos un QP paramétrico para cada rendimiento objetivo $\mu^*$:

$$
\min_{\mathbf{w}} \quad \mathbf{w}^\top \boldsymbol{\Sigma} \, \mathbf{w} \qquad \text{s.a.} \quad \boldsymbol{\mu}^\top \mathbf{w} = \mu^*, \quad \sum w_i = 1, \quad w_i \geq 0
$$

Usamos `cp.Parameter()` con `warm_start=True` para reutilizar el problema y resolver eficientemente.

In [ ]:
def calc_efficient_frontier(returns, n_points=150):
    """Frontera eficiente mediante QP paramétrico con CVXPY (DCP)."""
    n = returns.columns.size
    mu = returns.mean().values
    Sigma = np.cov(returns.T, ddof=0)

    w = cp.Variable(n)
    target_ret = cp.Parameter()
    risk = cp.quad_form(w, Sigma)
    objective = cp.Minimize(risk)
    constraints = [mu @ w == target_ret, cp.sum(w) == 1, w >= 0]
    prob = cp.Problem(objective, constraints)

    result_means, result_stds, result_weights = [], [], []
    for r in np.linspace(mu.min(), mu.max(), n_points):
        target_ret.value = r
        prob.solve(solver=cp.ECOS, warm_start=True)
        if prob.status == "optimal":
            result_means.append(round(r, 4))
            result_stds.append(round(np.sqrt(risk.value), 6))
            result_weights.append(np.round(w.value, 5))
    return {"Means": result_means, "Stds": result_stds, "Weights": result_weights}

frontier = calc_efficient_frontier(annual_returns)
print(f"Puntos en la frontera: {len(frontier['Means'])}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# Frontera eficiente
ax.plot(frontier["Stds"], frontier["Means"], "b-", linewidth=2, label="Frontera eficiente")

# Portafolio óptimo (max Sharpe)
port_std = np.sqrt(calc_portfolio_var(annual_returns, w_opt))
port_ret = annual_returns.mean().values @ w_opt
ax.scatter(port_std, port_ret, marker="*", s=300, color="red", zorder=5,
           label=f"Max Sharpe (S={sharpe_opt:.3f})")

# Activos individuales
for i, ticker in enumerate(tickers):
    ax.scatter(annual_returns[ticker].std(), annual_returns[ticker].mean(),
               marker="o", s=80, zorder=5)
    ax.annotate(ticker, (annual_returns[ticker].std(), annual_returns[ticker].mean()),
                fontsize=11, ha="left", va="bottom")

ax.set_title("Frontera eficiente de Markowitz")
ax.set_xlabel("Riesgo (desviación estándar)")
ax.set_ylabel("Rendimiento esperado")
ax.legend()
plt.tight_layout()

---
## 5. Análisis de ETFs y clustering

### Selección de activos mediante clustering

Cuando el universo de activos es grande, es útil **agrupar** activos similares antes de construir un portafolio. Usamos dos técnicas:
- **K-Means**: agrupa por rendimiento medio y desviación estándar.
- **Clustering jerárquico**: agrupa por similitud en la matriz de correlación.

In [ ]:
etf_tickers = ["PICK", "IBB", "XBI", "AMLP", "VGT", "RYE", "IEO", "LABU"]
etf_closes = get_historical_closes(etf_tickers, "2025-01-01", "2025-03-27")
etf_returns = np.log(etf_closes / etf_closes.shift(1)).dropna()

etf_stats = pd.DataFrame({
    "Media": etf_returns.mean(),
    "Std": etf_returns.std()
})
etf_stats

### K-Means clustering

Agrupamos los ETFs en 3 clusters según su perfil rendimiento-riesgo (media, desviación estándar).

In [ ]:
n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
etf_stats["Cluster"] = kmeans.fit_predict(etf_stats[["Media", "Std"]])

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(etf_stats["Media"], etf_stats["Std"],
                     c=etf_stats["Cluster"], cmap="Set1", s=100, zorder=5)
for ticker, row in etf_stats.iterrows():
    ax.annotate(ticker, (row["Media"], row["Std"]), fontsize=10, ha="left", va="bottom")
ax.set_xlabel("Rendimiento medio diario")
ax.set_ylabel("Desviación estándar diaria")
ax.set_title(f"K-Means clustering de ETFs ({n_clusters} clusters)")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()

### Clustering jerárquico

El **dendrograma** muestra la estructura de similitud entre ETFs basada en la correlación de Spearman. Activos en la misma rama se mueven de forma similar.

In [ ]:
corr_mat = etf_returns.corr(method="spearman")
Z = hac.linkage(1 - corr_mat, method="ward")

fig, ax = plt.subplots(figsize=(12, 6))
hac.dendrogram(Z, labels=corr_mat.columns.tolist(), ax=ax,
               leaf_rotation=45, leaf_font_size=12)
ax.set_title("Dendrograma de clustering jerárquico (correlación de Spearman)")
ax.set_ylabel("Distancia")
plt.tight_layout()

---

## Navegación del curso

← **Anterior**: Clase 3: Opciones financieras  
→ **Siguiente**: Clase 5: Covarianza robusta (Shrunk Covariance)

---

## 6. Referencias bibliográficas

- **Boyd, S. & Vandenberghe, L.** (2004). *Convex Optimization*. Cambridge University Press. — Cap. 4: Convex optimization problems.
- **Cont, R.** (2001). Empirical Properties of Asset Returns. *Quantitative Finance*, 1(2), 223–236.
- **Hull, J. C.** (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson. — Cap. 22: Estimating Volatilities and Correlations.
- **Luenberger, D. G.** (2013). *Investment Science* (2nd ed.). Oxford University Press. — Cap. 6–8: Mean-Variance Portfolio Theory.
- **Markowitz, H.** (1952). Portfolio Selection. *The Journal of Finance*, 7(1), 77–91.
- **Sharpe, W. F.** (1966). Mutual Fund Performance. *The Journal of Business*, 39(1), 119–138.
- **Sharpe, W. F.** (1964). Capital Asset Prices. *The Journal of Finance*, 19(3), 425–442.
- **Tsay, R. S.** (2010). *Analysis of Financial Time Series* (3rd ed.). Wiley. — Cap. 9: Portfolio Analysis.
- **Venegas Martínez, F.** (2008). *Riesgos financieros y económicos* (2a ed.). Cengage Learning.